# SMIXAE toy notebook

Iterative SMIXAE development on the toy model. Converted from the marimo notebook `toy_smixae.py`.

In [6]:
import sys
from pathlib import Path

import plotly.graph_objects as go
import torch
from IPython.display import Markdown, display
from plotly.subplots import make_subplots

# Ensure src/ is on the path (assumes the notebook is run from notebooks/)
_src = Path.cwd().parent / "src"
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

from collections.abc import Callable
from dataclasses import dataclass

from sae_lens.saes.batchtopk_sae import BatchTopK
from sae_lens.saes.sae import (
    SAEConfig,
    SAEMetadata,
    TrainCoefficientConfig,
    TrainingSAEConfig,
    TrainStepInput,
    TrainStepOutput,
)
from sae_lens.synthetic import train_toy_sae
from transformer_lens.hook_points import HookPoint
from typing_extensions import override

import smixae  # noqa: F401 -- registers architecture with SAELens
from smixae.base_smixae import (
    BaseSMIXAE,
    BaseSMIXAETraining,
    register_smixae_v1_bottleneck_weights,
    register_standard_linear_weights,
)
from toy.metrics import compute_cofiring_matrix, compute_metrics, compute_restricted_r2
from toy.plot import plot_all_experts_with_originals
from toy.zoo import (
    EvalData,
    ManifoldActivationGenerator,
    ManifoldZoo,
    build_manifold_zoo,
    generate_eval_set,
)

import einops as eo

from torch import nn
import torch.nn.functional as F

display(Markdown("## Imports loaded"))


## Imports loaded

In [2]:
# -- Dataset params --------------------------------------------------
seed        = 0
d_in        = 128
l0          = 4           # active manifolds per sample
sigma_bias  = 5.0

# -- Model architecture -----------------------------------------------
n_experts    = 48         # should match n_instances in zoo (48)
d_expert     = 16
d_bottleneck = 3          # 3-D for direct visualisation
d_sae = n_experts * d_expert

# -- Training -----------------------------------------------------------
k_experts         = 4     # active experts per sample (BatchTopK budget)
training_samples  = 50_000_000
batch_size        = 2048
lr                = 3e-3
lr_warm_up_steps  = 2_000
n_snapshots       = 20    # how many times compute_restricted_r2 is called during training

# -- Infrastructure -------------------------------------------------------
device       = "cuda" if torch.cuda.is_available() else "cpu"
toy_data_dir = Path("toy_data")

display(Markdown(f"""
**Config**

| Param | Value |
|---|---|
| seed | {seed} |
| d_in | {d_in} |
| l0 | {l0} |
| sigma_bias | {sigma_bias} |
| n_experts | {n_experts} |
| k_experts | {k_experts} |
| training_samples | {training_samples:,} |
| device | {device} |
"""))



**Config**

| Param | Value |
|---|---|
| seed | 0 |
| d_in | 128 |
| l0 | 4 |
| sigma_bias | 5.0 |
| n_experts | 48 |
| k_experts | 4 |
| training_samples | 50,000,000 |
| device | cuda |


In [3]:
def _dataset_dir(base, s, d, l0_val, b):
    return base / f"seed{s}_d{d}_l{l0_val}_b{b:g}"

_ds = _dataset_dir(toy_data_dir, seed, d_in, l0_val=l0, b=sigma_bias)
_zoo_path  = _ds / "zoo.pt"
_eval_path = _ds / "eval.pt"

if _zoo_path.exists() and _eval_path.exists():
    zoo       = ManifoldZoo.load(_zoo_path, device=device)
    eval_data = EvalData.load(_eval_path, device="cpu")
    _source   = f"Loaded from `{_ds}`"
else:
    _ds.mkdir(parents=True, exist_ok=True)
    zoo = build_manifold_zoo(d_in=d_in, seed=seed, device=device, sigma_bias=sigma_bias)
    zoo.save(_zoo_path)
    eval_data = generate_eval_set(zoo, n_samples=200_000, l0=l0, seed=seed + 1, device=device)
    eval_data.save(_eval_path)
    _source = f"Generated and saved to `{_ds}`"

_type_counts = {}
for _inst in zoo.instances:
    _type_counts[_inst.type_name] = _type_counts.get(_inst.type_name, 0) + 1

display(Markdown(f"""
**Toy Data** -- {_source}

- Zoo: {len(zoo.instances)} instances, {zoo.n_atoms} atoms, d_in={zoo.d_in}
- Eval set: {eval_data.x.shape[0]:,} samples of shape {tuple(eval_data.x.shape)}
- Manifold types: {dict(sorted(_type_counts.items()))}
"""))


2026-06-16 19:25:23.470 | INFO     | toy.zoo:optimize_subspaces:273 - Grassmannian optimisation: 48 subspaces in R^128
2026-06-16 19:25:23.490 | INFO     | toy.zoo:optimize_subspaces:274 -   max spectral coherence (before): 0.373155
2026-06-16 19:25:27.306 | INFO     | toy.zoo:optimize_subspaces:297 -   step  100/500  max_spectral_coh=0.006664  cost=0.140572
2026-06-16 19:25:30.725 | INFO     | toy.zoo:optimize_subspaces:297 -   step  200/500  max_spectral_coh=0.000033  cost=0.140564
2026-06-16 19:25:34.247 | INFO     | toy.zoo:optimize_subspaces:297 -   step  300/500  max_spectral_coh=0.000000  cost=0.140564
2026-06-16 19:25:37.770 | INFO     | toy.zoo:optimize_subspaces:297 -   step  400/500  max_spectral_coh=0.000000  cost=0.140564
2026-06-16 19:25:41.403 | INFO     | toy.zoo:optimize_subspaces:297 -   step  500/500  max_spectral_coh=0.000000  cost=0.140564
2026-06-16 19:25:41.413 | INFO     | toy.zoo:optimize_subspaces:305 -   max spectral coherence (after):  0.000000



**Toy Data** -- Generated and saved to `toy_data/seed0_d128_l4_b5`

- Zoo: 48 instances, 120 atoms, d_in=128
- Eval set: 200,000 samples of shape (200000, 128)
- Manifold types: {'circle': 6, 'flat_disk': 6, 'helix': 6, 'mobius': 6, 'segment': 6, 'sphere': 6, 'swiss_roll': 6, 'torus': 6}


In [4]:
# Woke JumpReLU code

# Zohran Mamdami piecewise linear function
def rectangle(x: torch.Tensor) -> torch.Tensor:
    return ((x > -0.5) & (x < 0.5)).to(x)

# Kamala Harris step function
class Step(torch.autograd.Function):
    @staticmethod
    def forward(
        x: torch.Tensor,
        threshold: torch.Tensor,
        bandwidth: float,  # noqa: ARG004
    ) -> torch.Tensor:
        return (x > threshold).to(x)

    @staticmethod
    def setup_context(
        ctx: Any, inputs: tuple[torch.Tensor, torch.Tensor, float], output: torch.Tensor
    ) -> None:
        x, threshold, bandwidth = inputs
        del output
        ctx.save_for_backward(x, threshold)
        ctx.bandwidth = bandwidth

    @staticmethod
    def backward(  # type: ignore[override]
        ctx: Any, grad_output: torch.Tensor
    ) -> tuple[None, torch.Tensor, None]:
        x, threshold = ctx.saved_tensors
        bandwidth = ctx.bandwidth
        threshold_grad = torch.sum(
            -(1.0 / bandwidth) * rectangle((x - threshold) / bandwidth) * grad_output,
            dim=0,
        )
        return None, threshold_grad, None
    
# Goal is to use the Step Function to create a polytope gate that decides whether to let in activations based on woke geometry rather than conservative linear direction ideals

### SMIXAERebased -- self-contained copy. Edit freely; does NOT affect the library.
Base classes and weight helpers are stable library imports -- edit the classes below.

In [11]:
import math
from einops.layers.torch import EinMix as Mix # GOAT of linear layers


# TODO: 
# Add proper initializaiton with options to the Tensor Linear Layer - probably fix it to be 3D + 2D bias
# Convert everything in the SMIXAE Class to that class - more clear
# Add geometric gate
# Add aux loss
# Add step function loss
# ???
# Profit

# Create a Gretchen Whitmer geometric gate layer
# No more Trump era hand definition of weight components
# Thank you for your attention to this matter
# class GeoGate(torch.nn.Module):
#     def __init__(self, n_experts : int, d_bottleneck : int, n_hidden_layers : int, d_layer : int, dtype : torch.dtype, device : torch.device, use_bias : bool = True, threshold_initialization_value : float = 0.01, leaky_relu_slope = 1e-4):
#         super.__init__()
        
#         # Activation function to use - use LeakyReLU because dead neurons are fascist
#         self.activation_function = torch.nn.LeakyReLU(negative_slope=leaky_relu_slope)

#         factory_kwargs = {'dtype' : dtype, 'device' : device}



#         # Define weight matrices, thanks Obama
#         weights = nn.ParameterList[
            
#             for _ in range(n_layers)
#         ]

        
#         self.layer_1 = nn.Paramter(
#             torch.empty(
#                 n_experts,
#                 d_bottleneck,

#                 **factory_kwargs,
#             )
#         )

#         # Define thresholds based on iterative sampling of NoKings protest count estimations

def register_smixae_v2_weights(sae : BaseSMIXAE | BaseSMIXAETraining):
    """
    Update weight configuration for SMIXAEv2.

    Key principles:
    - Remove unnneccessary additional linear layer
    - Encoders and decoder are 3-tensors - no flattening!
    - Bias is now a 2-tensor
    - Drag queen story hour level woke with Einops
    """
    dims = {
        'd_in' : sae.cfg.d_in,
        'n_experts' : sae.cfg.n_experts,
        'd_expert' : sae.cfg.d_expert,
        'd_bottleneck' : sae.cfg.d_bottleneck
    }

    sae.W_enc = Mix(
        pattern='... d_in -> ... n_experts d_expert',
        weight_shape='d_in n_experts d_expert',
        bias_shape='n_experts d_expert',
        **dims,
    )

    sae.W_bottleneck = Mix(
        pattern='... n_experts d_expert -> ... n_experts d_bottleneck',
        weight_shape='n_experts d_expert d_bottleneck',
        **dims,
    )

    sae.W_dec = Mix(
        pattern='... n_experts d_bottleneck -> ... d_in',
        weight_shape='n_experts d_bottleneck d_in',
        **dims,
    )

In [ ]:
# -- Shared encode -----------------------------------------------------------

def _smixae_rebased_encode(sae, x: torch.Tensor):  # noqa: ANN001
    """X -> W_enc + b_enc -> LeakyReLU -> (n_experts, d_expert) -> W_bottleneck -> bottleneck.

    Do not fear the einsums, they are just matmuls but for the 3-tensors.

    Returns (h_charts, pre_act_charts, pre_act_bottleneck).
    """
    sae_in = sae.process_sae_in(x)


    pre_act_charts = eo.einsum(sae_in, sae.W_enc, "batch_size d_in, d_in n_experts d_expert -> batch_size n_experts d_expert") + sae.b_enc
    h_charts = sae.activation_fn(pre_act_charts)

    pre_act_bottleneck = eo.einsum(h_charts, sae.W_bottleneck, "batch_size n_experts d_expert, n_experts d_expert d_bottleneck -> batch_size n_experts d_bottleneck")

    return pre_act_charts, h_charts, pre_act_bottleneck

def _smixae_decode(sae : "SMIXAERebased | SMIXAERebasedTraining", z : torch.Tensor) -> torch.Tensor:
    """
    Decode feature acts
    """
    # Divide by frob norm per expert, provides stability
    W_dec_normed = sae.W_dec / sae.effective_decoder_norm.view(-1, 1, 1)

    # Apply Decoder
    return eo.einsum(W_dec_normed, z, "n_experts d_bottleneck d_in, batch_size n_experts d_bottleneck -> batch_size d_in") + sae.b_dec

# -- Inference config + class --------------------------------------------------

@dataclass
class SMIXAERebasedConfig(SAEConfig):
    """Inference config."""

    n_experts: int = 1024
    d_expert: int = 16
    d_bottleneck: int = 3
    rescale_acts_by_decoder_norm: bool = True
    d_sae : int = 1024 * 3

    @override
    @classmethod
    def architecture(cls) -> str:
        return "smixae_nb"

class SMIXAERebased(BaseSMIXAE[SMIXAERebasedConfig]):
    """Inference-only SMIXAERebased."""

    @override
    def initialize_weights(self) -> None:
        register_smixae_v2_weights(self)

        self.register_buffer("threshold", torch.tensor(0.0, dtype=torch.double, device=self.device, requires_grad=False))
        self.register_buffer("n_passes_since_fired", torch.zeros(self.cfg.n_experts, dtype=torch.long))

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        _, _, pre_act_bottleneck = _smixae_rebased_encode(self, x)
        mask = pre_act_bottleneck.norm(dim=-1) > self.threshold  # type: ignore[operator]
        return (pre_act_bottleneck - self.b_select) * mask.unsqueeze(-1)

    def encode_with_charts(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        h_charts, _, pre_act_bottleneck = _smixae_rebased_encode(self, x)
        mask = pre_act_bottleneck.norm(dim=-1) > self.threshold  # type: ignore[operator]
        return (pre_act_bottleneck - self.b_select) * mask.unsqueeze(-1), h_charts

    def decode(self, feature_acts: torch.Tensor) -> torch.Tensor:
        out = _smixae_decode(self, feature_acts)

        out = self.hook_sae_recons(out)
        out = self.run_time_activation_norm_fn_out(out)
        return self.reshape_fn_out(out, self.d_head)

    def get_activation_fn(self) -> Callable[[torch.Tensor], torch.Tensor]:
        return torch.nn.LeakyReLU(negative_slope=1e-4)

    @override
    @torch.no_grad()
    def fold_activation_norm_scaling_factor(self, scaling_factor: float) -> None:
        self.W_enc.data *= scaling_factor
        self.W_dec.data /= scaling_factor
        self.b_dec.data /= scaling_factor
        self.cfg.normalize_activations = "none"
        if self.cfg.rescale_acts_by_decoder_norm:
            sf_sqrt = scaling_factor**0.5
            self.W_dec.data *= sf_sqrt
            self.threshold = self.threshold / sf_sqrt  # type: ignore[assignment]
        else:
            self.threshold = self.threshold / scaling_factor  # type: ignore[assignment]

    @property
    def effective_decoder_norm(self) -> torch.Tensor:
        W_dec_r = self.W_dec.view(self.cfg.n_experts, self.cfg.d_expert, -1)
        return torch.linalg.matrix_norm(self.W_latent_dec @ W_dec_r, ord="fro", dim=(-2, -1))

# -- Training config + class ---------------------------------------------------

@dataclass
class SMIXAERebasedTrainingConfig(TrainingSAEConfig):
    """Training config."""

    n_experts: int = 1024
    d_expert: int = 16
    d_bottleneck: int = 3
    k_experts: int = 8
    aux_loss_coefficient: float = 1 / 32
    rescale_acts_by_decoder_norm: bool = True
    threshold_lr: float = 0.1
    dead_after_n_passes: int = 1000
    d_sae : int = 1024 * 3

    @override
    @classmethod
    def architecture(cls) -> str:
        return "smixae_nb"

class SMIXAERebasedTraining(BaseSMIXAETraining[SMIXAERebasedTrainingConfig]):
    """Training SMIXAERebased -- edit this class to experiment.

    Key override points:
      training_forward_pass       -- change loss terms or add new ones
      calculate_pre_act_aux_loss  -- swap dead-expert recovery strategy
      encode_with_hidden_pre      -- change routing / bottleneck projection
      decode                      -- change reconstruction path
      _smixae_rebased_encode      -- change the core encode function above
    """

    def __init__(self, cfg: SMIXAERebasedTrainingConfig) -> None:
        cfg.d_sae = cfg.d_expert * cfg.n_experts
        super().__init__(cfg)
        self.hook_l0 = HookPoint()
        self.hook_sae_acts_bottleneck = HookPoint()
        self.batchtopk = BatchTopK(self.cfg.k_experts)

    @override
    def initialize_weights(self) -> None:
        # register_standard_linear_weights(self)
        # register_smixae_v1_bottleneck_weights(self)
        register_smixae_v2_weights(self)

        self.register_buffer("threshold", torch.tensor(0.0, dtype=torch.double, device=self.device))
        self.register_buffer("n_passes_since_fired", torch.zeros(self.cfg.n_experts, dtype=torch.long))

    @override
    def get_coefficients(self) -> dict[str, TrainCoefficientConfig | float]:
        return {}

    def get_activation_fn(self) -> Callable[[torch.Tensor], torch.Tensor]:
        return torch.nn.LeakyReLU(negative_slope=1e-4)

    @property
    def effective_decoder_norm(self) -> torch.Tensor:
        # W_dec_r = self.W_dec.view(self.cfg.n_experts, self.cfg.d_expert, -1)
        # return torch.linalg.matrix_norm(self.W_latent_dec @ W_dec_r, ord="fro", dim=(-2, -1))

        # Frobenius norm per expert, treating them as if they are separate matrices
        return eo.reduce(model.W_dec ** 2, "n_experts d_bottleneck d_in -> n_experts", 'sum') ** 0.5

    def encode_with_hidden_pre(self, x: torch.Tensor) -> tuple[torch.Tensor, ...]:
        pre_act_charts, h_charts, pre_act_bottleneck = _smixae_rebased_encode(self, x)

        # Apply BatchTopK
        batch_norm_mask = self.batchtopk(pre_act_bottleneck.norm(dim=-1)) > 0
        h_bottleneck = (pre_act_bottleneck - self.b_select) * batch_norm_mask.unsqueeze(-1)

        # Hooks and stashes
        self.h_bottleneck = h_bottleneck  # stored for compute_restricted_r2 / plot helpers
        self.hook_sae_acts_pre(pre_act_charts)
        self.hook_sae_acts_post(h_charts)
        self.hook_sae_acts_bottleneck(h_bottleneck)

        return h_bottleneck, pre_act_bottleneck, h_charts, pre_act_charts

    def decode(self, feature_acts: torch.Tensor) -> torch.Tensor:
        out = _smixae_decode(self, feature_acts)

        # Hooks
        out = self.hook_sae_recons(out)
        out = self.run_time_activation_norm_fn_out(out)

        return self.reshape_fn_out(out, self.d_head)

    @override
    def training_forward_pass(self, step_input: TrainStepInput) -> TrainStepOutput:
        # Encode
        # pre_act_charts, h_charts, pre_act_bottleneck = _smixae_rebased_encode(self, step_input.sae_in)
        h_bottleneck, pre_act_bottleneck, h_charts, pre_act_charts = self.encode_with_hidden_pre(step_input.sae_in)
        h_bottleneck_norms = h_bottleneck.norm(dim=-1)

        # ???
        # batch_norm_mask = self.batchtopk(pre_act_bottleneck.norm(dim=-1)) > 0
        # h_bottleneck = pre_act_bottleneck * batch_norm_mask.unsqueeze(-1)
        # self.h_bottleneck = h_bottleneck
        # self.hook_sae_acts_pre(pre_act_charts)
        # self.hook_sae_acts_post(h_charts)
        # self.hook_sae_acts_bottleneck(h_bottleneck)

        # Decode and update threshold
        sae_out = self.decode(h_bottleneck)
        self.update_threshold(h_bottleneck_norms)

        # Update the dead expert tracker
        with torch.no_grad():
            fired_in_batch = (h_bottleneck_norms > 0).any(dim=0)
            self.n_passes_since_fired = torch.where(
                fired_in_batch,
                torch.zeros_like(self.n_passes_since_fired),
                self.n_passes_since_fired + 1,
            )

        # Calculate MSE and dead expert aux loss
        mse_loss = self.mse_loss_fn(sae_out, step_input.sae_in).sum(dim=-1).mean()
        dead_aux_loss = self.calculate_pre_act_aux_loss(
            self.n_passes_since_fired > self.cfg.dead_after_n_passes,
            pre_act_bottleneck,
        )

        total_loss = mse_loss + dead_aux_loss
        losses = {"mse_loss": mse_loss, "dead_expert_aux_loss": dead_aux_loss}

        # WandB metrics
        metrics: dict = {
            "experts_above_1e-3_L2":   (h_bottleneck_norms > 1e-3).float().sum(dim=-1).mean(),
            "experts_above_1e-1_L2":   (h_bottleneck_norms > 1e-1).float().sum(dim=-1).mean(),
            "expert_norm_mean":        h_bottleneck_norms[h_bottleneck_norms > 0].mean(),
            "dead_experts":            (self.n_passes_since_fired > self.cfg.dead_after_n_passes).sum().item(),
            "act_threshold":           self.threshold,
            "experts_above_threshold": (pre_act_bottleneck.norm(dim=-1) > self.threshold).float().sum(dim=-1).mean(),
            "nonzero_latent_l0":       (h_charts > 0).float().sum(dim=-1).mean(),
        }

        # SAELens trainer expects (batch, d_sae) shape for its book-keeping.
        return TrainStepOutput(
            sae_in=step_input.sae_in, sae_out=sae_out,
            feature_acts=h_charts.flatten(start_dim=1), hidden_pre=pre_act_charts.flatten(start_dim=1),
            loss=total_loss, losses=losses, metrics=metrics,
        )

    def calculate_pre_act_aux_loss(
        self, dead_expert_mask: torch.Tensor, pre_act_bottleneck: torch.Tensor
    ) -> torch.Tensor:
        if dead_expert_mask is None or not dead_expert_mask.any():
            return self.threshold.new_tensor(0.0)
        expert_norms = pre_act_bottleneck.norm(dim=-1)
        dead_norms = expert_norms[:, dead_expert_mask]
        shortfall = torch.relu(self.threshold.detach().float() - dead_norms)

        return self.cfg.aux_loss_coefficient * (shortfall).sum(dim=-1).mean()

    @torch.no_grad()
    def update_threshold(self, norms_topk: torch.Tensor) -> None:
        positive_mask = norms_topk > 0
        lr = self.cfg.threshold_lr
        with torch.autocast(self.threshold.device.type, enabled=False):
            if positive_mask.any():
                min_positive = norms_topk[positive_mask].min().to(self.threshold.dtype)
                self.threshold = (1 - lr) * self.threshold + lr * min_positive  # type: ignore[assignment]

    @override
    @torch.no_grad()
    def fold_activation_norm_scaling_factor(self, scaling_factor: float) -> None:
        self.W_enc.data *= scaling_factor
        self.W_dec.data /= scaling_factor
        self.b_dec.data /= scaling_factor
        self.cfg.normalize_activations = "none"

        # Fold norm rescaling into the decoder
        self.W_dec.data /= self.effective_decoder_norm.view(-1, 1, 1)

        # if self.cfg.rescale_acts_by_decoder_norm:
        #     sf_sqrt = scaling_factor**0.5
        #     self.W_dec.data *= sf_sqrt
        #     self.threshold = self.threshold / sf_sqrt  # type: ignore[assignment]
        # else:
        #     self.threshold = self.threshold / scaling_factor  # type: ignore[assignment]

    @override
    def calculate_aux_loss(
        self,
        step_input: TrainStepInput,
        feature_acts: torch.Tensor,
        hidden_pre: torch.Tensor,
        sae_out: torch.Tensor,
    ) -> dict[str, torch.Tensor]:
        return {
            "dead_expert_aux_loss": self.calculate_pre_act_aux_loss(
                self.n_passes_since_fired > self.cfg.dead_after_n_passes,
                hidden_pre,
            )
        }

# -- Instantiate (mirrors _build_smixae in cli/toy.py) ------------------------

torch.manual_seed(seed)
model = SMIXAERebasedTraining(
    SMIXAERebasedTrainingConfig(
        d_in=zoo.d_in,
        n_experts=n_experts,
        d_expert=d_expert,
        d_sae=n_experts*d_expert,
        d_bottleneck=d_bottleneck,
        k_experts=k_experts,
        normalize_activations="none",
        apply_b_dec_to_input=False,
        device=device,
        dead_after_n_passes=200,
        metadata=SAEMetadata(model_name="synthetic_toy", hook_name="ambient"),
    )
).to(device)

_n_params = sum(p.numel() for p in model.parameters())
display(Markdown(f"**Model**: SMIXAERebasedTraining (local copy) -- {_n_params:,} parameters"))


In [ ]:
print(model.W_dec.shape)

model.W_dec.norm(dim=1).shape

a = eo.reduce(model.W_dec ** 2, "n_experts d_bottleneck d_in -> n_experts", 'sum') ** 0.5
a.shape

# print(eo.repeat(model.effective_decoder_norm, "n_experts -> n_experts i j"), i=1, j=1)
model.W_dec / model.effective_decoder_norm.view(-1, 1, 1)


In [ ]:
# History is collected via snapshot callbacks -- these are co-firing matched
# metrics, NOT the reconstruction R2 that SAELens reports internally.
history: dict[str, list] = {
    "sample":   [],
    "r2":       [],   # mean co-firing matched R2 across all instances
    "cofiring": [],   # mean co-firing rate of matched expert
    "mse":      [],
    "dead":     [],
}

_total_samples = training_samples

def _snapshot_fn(trainer) -> None:
    """Called n_snapshots times, evenly spaced during training."""
    _sae = trainer.sae
    _sae.eval()

    _r2, _cf, _ = compute_restricted_r2(_sae, zoo, eval_data, device=device)
    _m          = compute_metrics(_sae, zoo, eval_data, device=device)

    history["sample"].append(trainer.n_training_samples)
    history["r2"].append(float(_r2.mean()))
    history["cofiring"].append(float(_cf.mean()))
    history["mse"].append(_m["mse"])
    history["dead"].append(_m["dead_experts"])

    _sae.train()

_manifold_ag = ManifoldActivationGenerator(zoo=zoo, l0=l0, device=device)

_save_dir = toy_data_dir / "notebook_model"
_save_dir.mkdir(parents=True, exist_ok=True)

train_toy_sae(
    sae=model,
    feature_dict=zoo.feature_dict,
    activations_generator=_manifold_ag,
    training_samples=_total_samples,
    batch_size=batch_size,
    lr=lr,
    lr_warm_up_steps=lr_warm_up_steps,
    device=device,
    n_snapshots=n_snapshots,
    snapshot_fn=_snapshot_fn,
)
model.eval()

# Final evaluation
r2_final, cofiring_final, best_experts = compute_restricted_r2(model, zoo, eval_data, device=device)
metrics_final = compute_metrics(model, zoo, eval_data, device=device)
cofiring_matrix = compute_cofiring_matrix(model, zoo, eval_data, device=device)

display(Markdown(f"""
**Training complete**

| Metric | Value |
|---|---|
| Mean R2 (co-firing matched) | {r2_final.mean():.4f} |
| Mean co-firing rate | {cofiring_final.mean():.3f} |
| MSE | {metrics_final['mse']:.5f} |
| Effective L0 | {metrics_final['effective_l0']:.2f} |
| Dead experts | {metrics_final['dead_experts']} |
"""))


In [ ]:
assert history["sample"], "No snapshots recorded -- increase n_snapshots."

_fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Mean R2 (co-firing matched)", "Mean co-firing rate",
        "MSE", "Dead experts",
    ],
)

_xs = history["sample"]

_fig.add_trace(go.Scatter(x=_xs, y=history["r2"], mode="lines+markers", name="R2"), row=1, col=1)
_fig.add_trace(go.Scatter(x=_xs, y=history["cofiring"], mode="lines+markers", name="co-fire"), row=1, col=2)
_fig.add_trace(go.Scatter(x=_xs, y=history["mse"], mode="lines+markers", name="MSE"), row=2, col=1)
_fig.add_trace(go.Scatter(x=_xs, y=history["dead"], mode="lines+markers", name="dead"), row=2, col=2)

_fig.update_xaxes(title_text="Training samples")
_fig.update_layout(title="Training curves (co-firing matched metrics)", height=600, showlegend=False)
_fig


In [ ]:
import collections as _col

_type_r2: dict[str, list] = _col.defaultdict(list)
for _i, _inst in enumerate(zoo.instances):
    _type_r2[_inst.type_name].append(float(r2_final[_i]))

_types  = sorted(_type_r2)
_means  = [sum(_type_r2[t]) / len(_type_r2[t]) for t in _types]
_mins   = [min(_type_r2[t]) for t in _types]
_maxs   = [max(_type_r2[t]) for t in _types]

_fig_bar = go.Figure([
    go.Bar(
        name="mean R2", x=_types, y=_means,
        error_y=dict(
            type="data",
            symmetric=False,
            array=[_maxs[i] - _means[i] for i in range(len(_types))],
            arrayminus=[_means[i] - _mins[i] for i in range(len(_types))],
        ),
    )
])
_fig_bar.update_layout(
    title="R2 by manifold type (mean +/- min/max across variants)",
    yaxis_title="R2",
    xaxis_title="Manifold type",
    yaxis_range=[0, 1],
)
_fig_bar


In [ ]:
_labels = [f"{inst.type_name}[{inst.variant_idx}]" for inst in zoo.instances]
_fig_heat = go.Figure(go.Heatmap(
    z=cofiring_matrix.numpy(),
    x=[str(e) for e in range(cofiring_matrix.shape[1])],
    y=_labels,
    colorscale="Viridis",
    colorbar=dict(title="P(fire | active)"),
))
_fig_heat.update_layout(
    title="Co-firing matrix: P(expert e fires | manifold i active)",
    xaxis_title="Expert index",
    yaxis_title="Manifold instance",
    height=800,
)
_fig_heat


### All 48 experts: learned (red) vs original (blue)

In [ ]:
plot_all_experts_with_originals(
    model, zoo, eval_data, best_experts,
    k_experts=k_experts, r2=r2_final, cofiring=cofiring_final, device=device,
)
